In [ ]:
from google.colab import files
import pandas as pd
import io
uploaded = files.upload()

Saving dataset.csv to dataset.csv


In [ ]:
df = pd.read_csv('dataset.csv')
df.head()

,Time,Year,Month,Garment - Footwear - Hat_toDecCPI,CPI_toPM,CPI_rate
0,1.0,2015.0,1.0,100.7,100.700000,0.700000
1,2.0,2015.0,2.0,100.2,99.503476,-0.496524
2,3.0,2015.0,3.0,100.1,99.900200,-0.099800
3,4.0,2015.0,4.0,100.3,100.199800,0.199800
4,5.0,2015.0,5.0,100.3,100.000000,0.000000


In [ ]:
# === A1. Split data series to train and test (CPI Series 1) ===
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error
import numpy as np
import math

# 1️⃣ Load dataset
df = pd.read_csv("dataset.csv")

# 2️⃣ Select CPI series (Series 1)
cpi = df["CPI_toPM"]

# 3️⃣ Remove NaN values
cpi = cpi.dropna().astype(float).reset_index(drop=True)

# 4️⃣ Split into Train/Test (80% – 20%)
split_ratio = 0.8
split_index = int(len(cpi) * split_ratio)

train = cpi[:split_index]
test  = cpi[split_index:]

print("✅ A1. Split completed:")
print(f"Train size: {len(train)} observations")
print(f"Test size : {len(test)} observations")


✅ A1. Split completed:
Train size: 76 observations
Test size : 20 observations


In [ ]:
# === A2. Forecasting using ARIMA(0,0,1) ===
# Based on Step 1.4, best model: (pO,dO,qO) = (0,0,1)

# 1️⃣ Build and fit the model on train data
model = ARIMA(train, order=(0,0,1))
model_fit = model.fit()

# 2️⃣ Forecast for test period
forecast = model_fit.forecast(steps=len(test))

# 3️⃣ Evaluate model performance
rmse = math.sqrt(mean_squared_error(test, forecast))
print(f"✅ RMSE on test set: {rmse:.4f}")

# 4️⃣ Combine actual vs forecast for inspection
comparison = pd.DataFrame({
    "Actual": test.values,
    "Forecast": forecast.values
})
print(comparison.head())

# 5️⃣ Optional: show model summary
print(model_fit.summary())


✅ RMSE on test set: 0.1310
       Actual    Forecast
0   99.950050  100.018704
1   99.970015  100.023207
2   99.900020  100.023207
3  100.030024  100.023207
4   99.979990  100.023207
                               SARIMAX Results                                
Dep. Variable:               CPI_toPM   No. Observations:                   76
Model:                 ARIMA(0, 0, 1)   Log Likelihood                  16.378
Date:                Mon, 03 Nov 2025   AIC                            -26.755
Time:                        14:53:08   BIC                            -19.763
Sample:                             0   HQIC                           -23.961
                                 - 76                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const        100.0232      

**🔹 Interpretation of ARIMA(0,0,1) Results – Series 1 (CPI_toPM)**

1. Model Fit:

  * The selected model ARIMA(0,0,1) yields an AIC of –26.755 and BIC of –19.763, indicating a good fit compared to other candidate models (the lower the AIC, the better).

  * The RMSE on the test set is 0.1310, showing that the model has a small average forecasting error and performs well on unseen data.

2. Estimated Coefficients:

  * The constant term (const = 100.0232) reflects that the CPI index fluctuates around 100, consistent with its definition as a normalized index.

  * The MA(1) coefficient = –0.5651 is statistically significant (p < 0.001), suggesting a negative short-term autocorrelation: the current value is inversely affected by the previous month’s random shock.

  * The error variance (σ² = 0.0379) is low, indicating stable and consistent model residuals.

3. Diagnostic Checking:

  * Ljung–Box Q(1) = 0.12, Prob(Q) = 0.73 → no autocorrelation in residuals; the model adequately captures the data’s dynamics.

  * Heteroskedasticity (H) ≈ 1.07 → residual variance is approximately constant (homoskedastic).

  * Jarque–Bera (JB) = 12.39, p < 0.01 → residuals deviate slightly from normality (mild skewness and kurtosis), which is acceptable for macroeconomic time series.

4. Overall Assessment:

  * The ARIMA(0,0,1) model captures the monthly fluctuations of the garment CPI effectively, with low forecast error (RMSE ≈ 0.13) and no residual autocorrelation.

  * It can therefore be regarded as the best-fitting model for Series 1, confirming the parameter selection from Step 1.4.

  * This model provides a solid baseline before introducing seasonal extensions in the following SARIMA analysis.

In [ ]:
# === B1. Split data series to train and test (CPI Rate - Series 3) ===
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error
import numpy as np
import math

# 1️⃣ Load dataset
df = pd.read_csv("dataset.csv")

# 2️⃣ If CPI rate not available, compute from CPI
if "CPI_rate" in df.columns:
    cpi_rate = df["CPI_rate"]
else:
    # Compute rate = % change of CPI_toPM
    cpi_rate = df["CPI_toPM"].pct_change() * 100

# 3️⃣ Remove NaN values (first row will be NaN if computed by pct_change)
cpi_rate = cpi_rate.dropna().astype(float).reset_index(drop=True)

# 4️⃣ Split into Train/Test (80% – 20%)
split_ratio = 0.8
split_index = int(len(cpi_rate) * split_ratio)

train_rate = cpi_rate[:split_index]
test_rate  = cpi_rate[split_index:]

print("✅ B1. Split completed:")
print(f"Train size: {len(train_rate)} observations")
print(f"Test size : {len(test_rate)} observations")


✅ B1. Split completed:
Train size: 76 observations
Test size : 20 observations


In [ ]:
# === B2. Forecasting using ARIMA(0,0,1) for CPI rate ===
# Best model based on Step 1.4 and A2 results

# 1️⃣ Build and fit the model on train data
model_rate = ARIMA(train_rate, order=(0,0,1))
model_rate_fit = model_rate.fit()

# 2️⃣ Forecast for test period
forecast_rate = model_rate_fit.forecast(steps=len(test_rate))

# 3️⃣ Evaluate performance using RMSE
rmse_rate = math.sqrt(mean_squared_error(test_rate, forecast_rate))
print(f"✅ RMSE on test set: {rmse_rate:.4f}")

# 4️⃣ Combine actual vs forecast for comparison
comparison_rate = pd.DataFrame({
    "Actual": test_rate.values,
    "Forecast": forecast_rate.values
})
print(comparison_rate.head())

# 5️⃣ Optional: show model summary
print(model_rate_fit.summary())


✅ RMSE on test set: 0.1310
     Actual  Forecast
0 -0.049950  0.018704
1 -0.029985  0.023207
2 -0.099980  0.023207
3  0.030024  0.023207
4 -0.020010  0.023207
                               SARIMAX Results                                
Dep. Variable:               CPI_rate   No. Observations:                   76
Model:                 ARIMA(0, 0, 1)   Log Likelihood                  16.378
Date:                Mon, 03 Nov 2025   AIC                            -26.755
Time:                        14:54:57   BIC                            -19.763
Sample:                             0   HQIC                           -23.961
                                 - 76                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0232      0.011      2.200      0.

🔹 Interpretation of ARIMA(0,0,1) Results – Series 3 (CPI_rate)


1. Model Fit:


* The selected model ARIMA(0,0,1) produces an AIC of –26.755 and a BIC of –19.763, indicating a good model fit for the CPI rate series.


* The RMSE on the test set is 0.1310, which is low, showing that the model forecasts short-term changes in the CPI rate with high accuracy.




2. Estimated Coefficients:




* The constant term (const = 0.0232) represents the average month-to-month percentage change in the CPI, implying a very small but positive inflation rate over the period.


* The MA(1) coefficient = –0.5651 is statistically significant (p < 0.001), suggesting that the current rate is negatively influenced by the random shock of the previous period — a typical mean-reverting pattern.


* The error variance (σ² = 0.0379) is small, confirming that the residuals are stable and the model explains most of the variation.




3. Diagnostic Checking:




* Ljung–Box Q(1) = 0.12, Prob(Q) = 0.73 > 0.05 → residuals are not autocorrelated, indicating that the ARIMA(0,0,1) model sufficiently captures the time-series dynamics.


* Heteroskedasticity (H) ≈ 1.07 → no sign of heteroskedasticity; residual variance is approximately constant.


* Jarque–Bera (JB) = 12.39, p < 0.01 → residuals slightly deviate from the normal distribution (moderate skewness = 0.40, kurtosis = 4.81), which is acceptable for economic data.




4. Overall Assessment:




* The ARIMA(0,0,1) model provides a reliable representation of the monthly CPI rate changes, with low forecast error (RMSE ≈ 0.13) and well-behaved residuals.


* The results are consistent with those obtained for Series 1 (CPI_toPM), confirming that the first-order moving-average process (MA(1)) adequately characterizes both the level and rate of the CPI in the garment sector.


* Hence, ARIMA(0,0,1) is validated as the best model for the CPI rate in this forecasting experiment.



In [14]:
# === A1. SARIMA model forecasting for CPI (Series 1) ===
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error
import numpy as np
import math

# 1️⃣ Load dataset
df = pd.read_csv("dataset.csv")

# 2️⃣ Select CPI series and clean NaN
cpi = df["CPI_toPM"].dropna().astype(float).reset_index(drop=True)

# 3️⃣ Split data 80% train / 20% test
split_index = int(len(cpi) * 0.8)
train, test = cpi[:split_index], cpi[split_index:]

# 4️⃣ Define known optimal non-seasonal parameters (pO,dO,qO)
pO, dO, qO = 0, 0, 1

# 5️⃣ Define one combination of seasonal parameters (P,D,Q,m)
P, D, Q, m = 1, 0, 1, 12   # Example: (1,0,1,12)

# 6️⃣ Fit SARIMA model
model = SARIMAX(train,
                order=(pO, dO, qO),
                seasonal_order=(P, D, Q, m),
                enforce_stationarity=False,
                enforce_invertibility=False)
model_fit = model.fit()

# 7️⃣ Forecast for test period
forecast = model_fit.forecast(steps=len(test))

# 8️⃣ Evaluate performance
rmse = math.sqrt(mean_squared_error(test, forecast))
print(f"✅ SARIMA({pO},{dO},{qO})({P},{D},{Q},{m}) RMSE: {rmse:.4f}")

# 9️⃣ Compare actual vs forecast
comparison = pd.DataFrame({
    "Actual": test.values,
    "Forecast": forecast.values
})
print(comparison.head())

# 🔟 Optional: show model summary
print(model_fit.summary())


✅ SARIMA(0,0,1)(1,0,1,12) RMSE: 0.2543
       Actual    Forecast
0   99.950050  100.354786
1   99.970015   99.780333
2   99.900020  100.238073
3  100.030024   99.904262
4   99.979990  100.164279
                                     SARIMAX Results                                      
Dep. Variable:                           CPI_toPM   No. Observations:                   76
Model:             SARIMAX(0, 0, 1)x(1, 0, 1, 12)   Log Likelihood                 -14.483
Date:                            Mon, 03 Nov 2025   AIC                             36.966
Time:                                    15:27:25   BIC                             45.474
Sample:                                         0   HQIC                            40.306
                                             - 76                                         
Covariance Type:                              opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975

🔹 Interpretation of SARIMA(0,0,1)(1,0,1,12) Results – Series 1 (CPI_toPM)

1. Model Fit:

* The SARIMA(0,0,1)(1,0,1,12) model gives an AIC = 36.966 and BIC = 45.474, both higher than the previous ARIMA(0,0,1) model (AIC = –26.755), indicating a weaker overall fit.

* The RMSE = 0.2543 on the test set is roughly twice that of the ARIMA model (0.1310), suggesting that the seasonal extension does not improve forecasting accuracy for this series.

2. Estimated Coefficients:

* MA(1) = –0.5974 is statistically significant (p < 0.001), confirming a negative short-term autocorrelation similar to the ARIMA case.

* AR.S.L12 = 1.0000 represents a seasonal autoregressive component with a 12-month lag; its value of 1.0 indicates a strong persistent seasonal pattern, although the extremely large z-value reflects numerical instability.

* MA.S.L12 = 6.3190, with p = 0.226, is not statistically significant, implying that the seasonal moving-average term contributes little explanatory power.

* The error variance (σ² = 0.0028) is small, but the model’s higher RMSE suggests that additional seasonal parameters may cause overfitting.

3. Diagnostic Checking:

* Ljung–Box Q(1) = 0.14, Prob(Q) = 0.71 > 0.05 → residuals are not autocorrelated; the model adequately captures serial dependence.

* Heteroskedasticity (H) = 3.57 > 1 indicates potential non-constant variance in residuals, suggesting mild heteroskedasticity.

* Jarque–Bera (JB) = 0.18, p = 0.91 > 0.05 → residuals are approximately normally distributed, showing better normality than in the ARIMA model.

4. Overall Assessment:

* The inclusion of seasonal components does not improve model performance for CPI_toPM; RMSE increases and AIC/BIC values worsen.

* Despite capturing potential yearly periodicity, the seasonal terms are mostly insignificant, implying that the garment CPI index shows limited seasonality in this dataset.

* Therefore, ARIMA(0,0,1) remains the more efficient and parsimonious forecasting model for Series 1.

In [15]:
# === A2. Search for best seasonal (P,Q) combination by RMSE ===
import itertools

# 1️⃣ Define search space for P and Q (fix D=0, m=12)
P_values = [0, 1, 2]
Q_values = [0, 1, 2]
D, m = 0, 12
pO, dO, qO = 0, 0, 1

best_rmse = float("inf")
best_order = None
results = []

# 2️⃣ Loop over combinations of (P,Q)
for P, Q in itertools.product(P_values, Q_values):
    try:
        model = SARIMAX(train,
                        order=(pO, dO, qO),
                        seasonal_order=(P, D, Q, m),
                        enforce_stationarity=False,
                        enforce_invertibility=False)
        model_fit = model.fit(disp=False)
        forecast = model_fit.forecast(steps=len(test))
        rmse = math.sqrt(mean_squared_error(test, forecast))
        results.append((P, Q, rmse))

        print(f"SARIMA({pO},{dO},{qO})({P},{D},{Q},{m}) → RMSE: {rmse:.4f}")

        if rmse < best_rmse:
            best_rmse = rmse
            best_order = (P, D, Q, m)
    except Exception as e:
        print(f"Failed for (P={P}, Q={Q}): {e}")
        continue

# 3️⃣ Display best model
print("\n✅ Best seasonal model:")
print(f"SARIMA({pO},{dO},{qO}){best_order} with RMSE = {best_rmse:.4f}")

# 4️⃣ Refit best model for final summary
best_model = SARIMAX(train,
                     order=(pO, dO, qO),
                     seasonal_order=best_order,
                     enforce_stationarity=False,
                     enforce_invertibility=False)
best_fit = best_model.fit()
print(best_fit.summary())


SARIMA(0,0,1)(0,0,0,12) → RMSE: 98.1884
SARIMA(0,0,1)(0,0,1,12) → RMSE: 72.9781
SARIMA(0,0,1)(0,0,2,12) → RMSE: 86.9502
SARIMA(0,0,1)(1,0,0,12) → RMSE: 0.2280
SARIMA(0,0,1)(1,0,1,12) → RMSE: 0.2543
SARIMA(0,0,1)(1,0,2,12) → RMSE: 0.1692
SARIMA(0,0,1)(2,0,0,12) → RMSE: 0.2152
SARIMA(0,0,1)(2,0,1,12) → RMSE: 0.2105
SARIMA(0,0,1)(2,0,2,12) → RMSE: 0.2202

✅ Best seasonal model:
SARIMA(0,0,1)(1, 0, 2, 12) with RMSE = 0.1692
                                        SARIMAX Results                                        
Dep. Variable:                                CPI_toPM   No. Observations:                   76
Model:             SARIMAX(0, 0, 1)x(1, 0, [1, 2], 12)   Log Likelihood                   5.305
Date:                                 Mon, 03 Nov 2025   AIC                             -0.611
Time:                                         15:27:42   BIC                              8.950
Sample:                                              0   HQIC                             3.030


🔹 Interpretation of SARIMA(0,0,1)(1,0,2,12) Results – Series 1 (CPI_toPM)

1. Model Fit:

* The selected seasonal model SARIMA(0,0,1)(1,0,2,12) achieves the lowest RMSE = 0.1692, outperforming other seasonal configurations tested.

* The AIC = –0.611 and BIC = 8.950, although higher than those of the non-seasonal ARIMA(0,0,1) model, indicate that the inclusion of seasonal terms improves predictive accuracy on the test set while keeping the model relatively parsimonious.

2. Estimated Coefficients:

* MA(1) = –2.0353 is statistically significant (p < 0.001), confirming a strong short-term negative correlation in the residual structure.

* AR.S.L12 = 0.9999 (p < 0.001) represents a dominant seasonal autoregressive component, indicating a clear annual (12-month) cyclical behavior in the CPI pattern.

* MA.S.L12 = –1.1930 and MA.S.L24 = 0.2831 are not statistically significant (p > 0.05), suggesting that the 12- and 24-month moving-average effects are weak and may primarily fine-tune model smoothness rather than add predictive power.

* The error variance (σ² = 0.0074) remains low, showing stable residual variation across months.

3. Diagnostic Checking:

* Ljung–Box Q(1) = 0.45, Prob(Q) = 0.50 > 0.05 → no significant residual autocorrelation; the model adequately captures temporal dependence.

* Heteroskedasticity (H) ≈ 0.94 → residuals display homoskedasticity (constant variance).

* Jarque–Bera (JB) = 1.56, p = 0.46 > 0.05 → residuals follow an approximately normal distribution with minimal skewness (–0.09) and moderate kurtosis (3.85).

4. Overall Assessment:

* Among all seasonal variants, SARIMA(0,0,1)(1,0,2,12) provides the best balance between accuracy and parsimony, reducing forecast error (RMSE ≈ 0.17) compared with other SARIMA models.

* The significant MA(1) and strong seasonal AR(12) terms confirm the presence of both short-term corrections and yearly cycles in the garment CPI series.

* This model demonstrates that incorporating limited seasonality improves performance compared with simpler seasonal forms, though ARIMA(0,0,1) remains a strong non-seasonal baseline for comparison.

In [17]:
# === B1. SARIMA model forecasting for CPI rate (Series 3) ===
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error
import numpy as np
import math

# 1️⃣ Load dataset
df = pd.read_csv("dataset.csv")

# 2️⃣ Select CPI rate series or compute from CPI
if "CPI_rate" in df.columns:
    cpi_rate = df["CPI_rate"]
else:
    # compute as monthly % change of CPI_toPM
    cpi_rate = df["CPI_toPM"].pct_change() * 100

# 3️⃣ Remove NaN values
cpi_rate = cpi_rate.dropna().astype(float).reset_index(drop=True)

# 4️⃣ Split data 80% train / 20% test
split_index = int(len(cpi_rate) * 0.8)
train_rate, test_rate = cpi_rate[:split_index], cpi_rate[split_index:]

# 5️⃣ Define optimal non-seasonal parameters from Step 1.4
pO, dO, qO = 0, 0, 1

# 6️⃣ Define one seasonal combination
P, D, Q, m = 1, 0, 1, 12

# 7️⃣ Fit SARIMA model
model_rate = SARIMAX(train_rate,
                     order=(pO, dO, qO),
                     seasonal_order=(P, D, Q, m),
                     enforce_stationarity=False,
                     enforce_invertibility=False)
model_rate_fit = model_rate.fit()

# 8️⃣ Forecast for test period
forecast_rate = model_rate_fit.forecast(steps=len(test_rate))

# 9️⃣ Compute RMSE
rmse_rate = math.sqrt(mean_squared_error(test_rate, forecast_rate))
print(f"✅ SARIMA({pO},{dO},{qO})({P},{D},{Q},{m}) RMSE: {rmse_rate:.4f}")

# 🔟 Compare actual vs forecast
comparison_rate = pd.DataFrame({
    "Actual": test_rate.values,
    "Forecast": forecast_rate.values
})
print(comparison_rate.head())

# ⓫ Optional: show model summary
print(model_rate_fit.summary())


✅ SARIMA(0,0,1)(1,0,1,12) RMSE: 0.1298
     Actual  Forecast
0 -0.049950 -0.061303
1 -0.029985  0.011756
2 -0.099980 -0.013032
3  0.030024  0.006936
4 -0.020010 -0.009975
                                     SARIMAX Results                                      
Dep. Variable:                           CPI_rate   No. Observations:                   76
Model:             SARIMAX(0, 0, 1)x(1, 0, 1, 12)   Log Likelihood                  15.026
Date:                            Mon, 03 Nov 2025   AIC                            -22.052
Time:                                    15:29:06   BIC                            -13.543
Sample:                                         0   HQIC                           -18.711
                                             - 76                                         
Covariance Type:                              opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------

🔹 Interpretation of SARIMA(0,0,1)(1,0,1,12) Results – Series 3 (CPI_rate)

1. Model Fit:

* The SARIMA(0,0,1)(1,0,1,12) model achieves an AIC = –22.052 and BIC = –13.543, suggesting a reasonable fit for the CPI rate series.

* The RMSE = 0.1298 on the test set is slightly lower than that of the ARIMA(0,0,1) model (0.1310), indicating a small but measurable improvement in forecasting performance when seasonal effects are included.

2. Estimated Coefficients:

* The MA(1) = –0.3782 term is statistically significant (p < 0.001), confirming a negative short-term autocorrelation — similar to the ARIMA results.

* The AR.S.L12 = –0.0969 and MA.S.L12 = 0.0256 coefficients are not statistically significant (p > 0.05), implying that seasonal effects (12-month lag) are weak for the CPI rate series.

* The error variance (σ² = 0.0360) remains low, consistent with stable residual variation and reliable model estimation.

3. Diagnostic Checking:

* Ljung–Box Q(1) = 1.03, Prob(Q) = 0.31 > 0.05 → residuals are not autocorrelated, confirming that the model adequately captures the data’s dynamics.

* Heteroskedasticity (H) = 4.60 > 1 → some signs of non-constant variance, suggesting mild heteroskedasticity in residuals.

* Jarque–Bera (JB) = 11.79, p < 0.01 → residuals deviate slightly from normality (kurtosis = 5.13), which is acceptable in macroeconomic time series analysis.

4. Overall Assessment:

* The SARIMA(0,0,1)(1,0,1,12) model slightly improves forecast accuracy compared with the simpler ARIMA(0,0,1) model (RMSE reduced from 0.1310 to 0.1298).

* However, the seasonal coefficients are not significant, suggesting that seasonal effects are minor in the CPI rate series.

* The model confirms that CPI rate fluctuations are primarily driven by short-term moving-average behavior rather than strong annual seasonality.

* Therefore, while SARIMA(0,0,1)(1,0,1,12) offers marginal accuracy gains, ARIMA(0,0,1) remains a sufficient and parsimonious choice for CPI rate forecasting.*văn bản in nghiêng*

In [18]:
# === B2. Iterative search for best seasonal (P,Q) combination by RMSE ===
import itertools

# 1️⃣ Define search space
P_values = [0, 1, 2]
Q_values = [0, 1, 2]
D, m = 0, 12
pO, dO, qO = 0, 0, 1

best_rmse = float("inf")
best_order = None
results = []

# 2️⃣ Try all combinations of (P,Q)
for P, Q in itertools.product(P_values, Q_values):
    try:
        model = SARIMAX(train_rate,
                        order=(pO, dO, qO),
                        seasonal_order=(P, D, Q, m),
                        enforce_stationarity=False,
                        enforce_invertibility=False)
        model_fit = model.fit(disp=False)
        forecast = model_fit.forecast(steps=len(test_rate))
        rmse = math.sqrt(mean_squared_error(test_rate, forecast))
        results.append((P, Q, rmse))

        print(f"SARIMA({pO},{dO},{qO})({P},{D},{Q},{m}) → RMSE: {rmse:.4f}")

        if rmse < best_rmse:
            best_rmse = rmse
            best_order = (P, D, Q, m)
    except Exception as e:
        print(f"⚠️ Failed for (P={P}, Q={Q}): {e}")
        continue

# 3️⃣ Display best seasonal model
print("\n✅ Best seasonal SARIMA model for CPI rate:")
print(f"SARIMA({pO},{dO},{qO}){best_order} with RMSE = {best_rmse:.4f}")

# 4️⃣ Refit best model and print summary
best_model_rate = SARIMAX(train_rate,
                          order=(pO, dO, qO),
                          seasonal_order=best_order,
                          enforce_stationarity=False,
                          enforce_invertibility=False)
best_fit_rate = best_model_rate.fit()
print(best_fit_rate.summary())


SARIMA(0,0,1)(0,0,0,12) → RMSE: 0.1307
SARIMA(0,0,1)(0,0,1,12) → RMSE: 0.1301
SARIMA(0,0,1)(0,0,2,12) → RMSE: 0.1338
SARIMA(0,0,1)(1,0,0,12) → RMSE: 0.1314
SARIMA(0,0,1)(1,0,1,12) → RMSE: 0.1298
SARIMA(0,0,1)(1,0,2,12) → RMSE: 0.1536
SARIMA(0,0,1)(2,0,0,12) → RMSE: 0.1390
SARIMA(0,0,1)(2,0,1,12) → RMSE: 0.1383
SARIMA(0,0,1)(2,0,2,12) → RMSE: 0.1781

✅ Best seasonal SARIMA model for CPI rate:
SARIMA(0,0,1)(1, 0, 1, 12) with RMSE = 0.1298
                                     SARIMAX Results                                      
Dep. Variable:                           CPI_rate   No. Observations:                   76
Model:             SARIMAX(0, 0, 1)x(1, 0, 1, 12)   Log Likelihood                  15.026
Date:                            Mon, 03 Nov 2025   AIC                            -22.052
Time:                                    15:29:26   BIC                            -13.543
Sample:                                         0   HQIC                           -18.711
             

🔹 Interpretation of SARIMA(0,0,1)(1,0,1,12) Results – Series 3 (CPI_rate)

1. Model Fit:

* The best seasonal model for the CPI rate is SARIMA(0,0,1)(1,0,1,12) with an RMSE = 0.1298, slightly outperforming other seasonal configurations tested.

* The model yields an AIC = –22.052 and BIC = –13.543, confirming a reasonably good fit to the data.

* The improvement in RMSE compared with the non-seasonal ARIMA(0,0,1) (0.1310 → 0.1298) indicates a marginal gain from adding seasonal components.

2. Estimated Coefficients:

* The MA(1) = –0.3782 term is statistically significant (p < 0.001), reflecting a short-term negative correlation in the CPI rate dynamics.

* The seasonal AR(12) = –0.0969 and seasonal MA(12) = 0.0256 terms are not significant (p > 0.05), showing that yearly seasonal effects are weak or minimal.

* The error variance (σ² = 0.0360) is small, suggesting that the model residuals are stable and variance remains low.

3. Diagnostic Checking:

* Ljung–Box Q(1) = 1.03, Prob(Q) = 0.31 > 0.05 → residuals are not autocorrelated, confirming adequate model specification.

* Heteroskedasticity (H) = 4.60 > 1 → some heteroskedasticity is present, implying slight non-constant variance in the residuals.

* Jarque–Bera (JB) = 11.79, p < 0.01 → residuals deviate moderately from normality (kurtosis = 5.13), which is acceptable for macroeconomic time-series data.

4. Overall Assessment:

* The SARIMA(0,0,1)(1,0,1,12) model slightly improves predictive accuracy compared with the simpler ARIMA(0,0,1), though seasonal parameters are statistically insignificant.

* This indicates that CPI rate variations are mainly short-term, with minimal yearly cyclicality.

* Consequently, while SARIMA provides a small improvement in RMSE, the non-seasonal ARIMA(0,0,1) remains an equally effective and more parsimonious model for forecasting the CPI rate.